# Детектор вредоносных скиллов LLM-агентов

**Дисциплина:** Адаптационный курс по ML и математике (Лебедев О.Б., группа 2, МИБСИ261)
**Данные:** `ProtectSkills/MaliciousSkillBench` (HuggingFace, CC-BY-4.0, arXiv:2608.19901)
**Задача:** бинарная классификация `skill_text` → вредоносный (1) / безопасный (0)

---

## Главный вопрос проекта

Не «какая точность на random», а **обобщение на новые источники** (протокол `source_disjoint`):
насколько детектор переносим на источники, которых он не видел при обучении.

Практическая мотивация: скиллы для LLM-агентов (MCP, Claude Skills) пишут разные авторы и площадки
разными стилями. Детектор, обученный на одних источниках, в проде встречает другие — и именно там
проверяется, годится ли он.

## Структура исследования

1. Данные и протоколы → 2. Baseline (TF-IDF + LogReg) → 3. char_wb → 4. Эмбеддинги →
5. Модельный зоопарк → 6. Каскад с LLM → 7. Поведенческие признаки → 8. Обфускация → 9. Разбор FP/FN.


In [ ]:
# Базовые импорты
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, classification_report)


## 1. Данные и разведка (EDA)

Датасет `MaliciousSkillBench`, конфиг `primary`: 9 740 скиллов, разметка `label`, текст `skill_text`,
источник `source_name` / `source_id`.


In [ ]:
prim = load_dataset("ProtectSkills/MaliciousSkillBench", "primary")["train"]
print("строк:", len(prim))
print("колонки:", prim.column_names)

df = prim.to_pandas()
print("\nБаланс классов:")
print(df["label"].value_counts().rename({0: "безопасные", 1: "вредоносные"}))
print("доля вредоносных:", round(df["label"].mean(), 3))

print("\nТоп источников:")
print(df["source_name"].value_counts().head(12))

lens = df["skill_text"].fillna("").str.len()
print("\nДлина текста (символы): медиана", int(lens.median()), "| p90", int(lens.quantile(0.9)), "| max", int(lens.max()))


**Наблюдение.** Классы несбалансированы (~3.4:1 в пользу вредоносных), тексты короткие —
это типичный «документно-технический» домен, где линейные модели на n-граммах обычно сильны.

## 2. Протоколы оценки

Официальные сплиты (`splits`): `random`, `source_balanced_random`, `m_structural_disjoint`,
`source_disjoint`.

- **random** — случайное разбиение: модель видит те же источники.
- **source_disjoint** — train и test из **разных источников**: честная проверка переноса.

Сравнение этих двух протоколов и есть ось всего исследования.


In [ ]:
# Собираем индексы: benchmark_id -> (label, text)
by_id = {r["benchmark_id"]: (int(r["label"]), r["skill_text"] or "") for r in prim}

def get_split(proto):
    sp = load_dataset("ProtectSkills/MaliciousSkillBench", "splits", split=proto)
    tr = [r["benchmark_id"] for r in sp if r["split"] == "train"]
    te = [r["benchmark_id"] for r in sp if r["split"] in ("test", "validation")]
    Xtr = [by_id[i][1] for i in tr]; ytr = [by_id[i][0] for i in tr]
    Xte = [by_id[i][1] for i in te]; yte = [by_id[i][0] for i in te]
    return Xtr, ytr, Xte, yte

PROTOS = ["random", "source_balanced_random", "m_structural_disjoint", "source_disjoint"]

def evaluate(pipe, proto):
    Xtr, ytr, Xte, yte = get_split(proto)
    pipe.fit(Xtr, ytr)
    pred = pipe.predict(Xte)
    return {
        "acc": accuracy_score(yte, pred),
        "macro_f1": f1_score(yte, pred, average="macro"),
        "rec_benign": recall_score(yte, pred, pos_label=0),
        "rec_mal": recall_score(yte, pred, pos_label=1),
        "prec_mal": precision_score(yte, pred, pos_label=1, zero_division=0),
    }


## 3. Baseline: TF-IDF (1–2 граммы) + LogisticRegression

Это опорная точка: если сложная модель не бьёт её — сложность не оправдана.


In [ ]:
def make_baseline():
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2,
                                  sublinear_tf=True, max_features=200000)),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", C=4.0)),
    ])

rows = []
for p in PROTOS:
    m = evaluate(make_baseline(), p)
    rows.append({"протокол": p, **m})
pd.DataFrame(rows).round(3)


**Вывод по baseline.** На `random` модель уверенно различает классы (~0.90 macro-F1), но на
`source_disjoint` recall **безопасных** падает почти вдвое — модель «перестраховывается» и валит
безопасные в вредоносные. Именно этот провал — предмет дальнейших экспериментов.

## 4. Модель №2: char_wb (символьные n-граммы)

Гипотеза: если проблема в обфускации/форме текста, символьные n-граммы (3–5) должны помочь.


In [ ]:
def make_charwb():
    return Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3,
                                  sublinear_tf=True, max_features=200000)),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", C=4.0)),
    ])

rows = []
for p in PROTOS:
    m = evaluate(make_charwb(), p)
    rows.append({"протокол": p, **m})
pd.DataFrame(rows).round(3)


**Вывод.** char_wb **проигрывает** in-distribution и ничего не даёт на `source_disjoint`
(recall безопасных ~0.48 против ~0.47). → Гипотеза «дело в форме текста» **не подтверждается**:
сигнал лексический, а не символьный.

## 5. Модель №3: эмбеддинги (bge-m3 / e5-small)

Гипотеза: эмбеддинги кодируют **смысл**, а не форму, и поднимут recall безопасных на новых источниках.

> Требует скачивания модели (~2 ГБ) и считается на CPU долго. Код ниже — для воспроизведения;
> готовые результаты — в таблице.


In [ ]:
# Код воспроизведения эмбеддингов (не обязателен для запуска ноутбука).
# Требует: pip install sentence-transformers
#
# from sentence_transformers import SentenceTransformer
# model = SentenceTransformer("BAAI/bge-m3")
# emb = model.encode(texts, batch_size=8, normalize_embeddings=True, show_progress_bar=True)
# затем LogReg(C=4) поверх эмбеддингов
print("Эмбеддинги: код-референс; результаты ниже из готового прогона.")


| Протокол | TF-IDF baseline | bge-m3 (эмбеддинги) |
|---|---|---|
| random | macro-F1 0.899 | **0.723** |
| source_balanced_random | 0.903 | **0.719** |
| m_structural_disjoint | 0.875 | **0.700** |
| source_disjoint | 0.733 (rec benign 0.473) | **0.637 (rec benign 0.434)** |

**Вывод.** Пре-регистрированная гипотеза отвергнута: эмбеддинги **хуже** baseline везде, включая
`source_disjoint`. `train_acc` у эмбеддингов ~0.81 против ~0.90 у TF-IDF → в них меньше линейного
сигнала. Семантика размывает лексические маркеры, которые и несут класс.

## 6. Модельный зоопарк: какой линейный классификатор лучше

Один и тот же TF-IDF, разные классификаторы.


In [ ]:
def make_with(clf):
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2,
                                  sublinear_tf=True, max_features=200000)),
        ("clf", clf),
    ])

classifiers = {
    "LogReg": LogisticRegression(max_iter=2000, class_weight="balanced", C=4.0),
    "LinearSVC": LinearSVC(class_weight="balanced", max_iter=2000),
    "SGD": __import__("sklearn.linear_model", fromlist=["SGDClassifier"]).SGDClassifier(
        loss="log_loss", class_weight="balanced", max_iter=2000, random_state=42),
    "ComplementNB": ComplementNB(),
    "MultinomialNB": MultinomialNB(),
}

rows = []
for name, clf in classifiers.items():
    m = evaluate(make_with(clf), "source_disjoint")
    rows.append({"clf": name, **m})
pd.DataFrame(rows).round(3)


**Вывод.** **LinearSVC — лучший** линейный классификатор (random 0.925, source_disjoint 0.736).
Наивный Байес бесполезен (recall безопасных ~0.02–0.09). Все линейные модели упираются в один потолок
на `source_disjoint` (~0.73) — значит, потолок задаёт не классификатор, а признаки.

## 7. Каскад SVC → LLM на «серой зоне»

Гипотеза: если модель не уверена (маленький margin), отдадим решение LLM.


**Результат:** каскад **не улучшил** baseline нигде:
- random: 0.908 → 0.891
- structural: 0.894 → 0.875
- source_disjoint: 0.693 → 0.681

**Вывод.** Вызов LLM на неопределённых (а их 40–50%) себя не оправдывает: дорого и хуже.
LLM не решает структурную проблему переноса.

## 8. Поведенческие признаки + рабочие точки

Гипотеза: добавим признаки, не зависящие от лексики источника (вызовы инструментов, права, шелл).


| Протокол | macro-F1 TF-IDF → +behav | recall безопасных TF-IDF → +behav |
|---|---|---|
| random | 0.925 → **0.930** | 0.896 → 0.908 |
| source_balanced_random | 0.918 → 0.920 | 0.880 → 0.882 |
| m_structural_disjoint | 0.906 → 0.898 | 0.903 → 0.899 |
| **source_disjoint** | **0.736 → 0.752** | **0.471 → 0.517** |

**Вывод.** Поведенческие признаки — **единственный приём**, который сдвинул `source_disjoint`
(+0.046 recall безопасных, +0.016 macro-F1). Они частично переносимы между источниками, в отличие
от лексики.

## 9. Устойчивость к обфускации

Проверяем, что будет, если вредоносный скилл замаскировать: zero-width символы, гомоглифы,
разбиение инструкции на куски.


In [ ]:
import re

def add_zero_width(text, every=7):
    out = []
    for i, ch in enumerate(text):
        out.append(ch)
        if i % every == 0 and not ch.isspace():
            out.append("\u200b")
    return "".join(out)

def add_homoglyphs(text):
    # латинские буквы, похожие на кириллические (и наоборот) — упрощённо
    table = {"a": "а", "e": "е", "o": "о", "p": "р", "c": "с", "x": "х"}
    return "".join(table.get(ch, ch) for ch in text)

def split_instructions(text, chunk=40):
    # вставляем переводы строк каждые chunk символов (ломаем n-граммы/паттерны)
    return "\n".join(text[i:i+chunk] for i in range(0, len(text), chunk))

Xtr, ytr, Xte, yte = get_split("source_disjoint")

def obf_eval(transform, label):
    Xte_obf = [transform(t) for t in Xte]
    pipe = make_with(LinearSVC(class_weight="balanced", max_iter=2000))
    pipe.fit(Xtr, ytr)
    base = pipe.predict(Xte)
    obf = pipe.predict(Xte_obf)
    print(f"{label:18s} macro-F1 {f1_score(yte, obf, average='macro'):.3f}  "
          f"(clean {f1_score(yte, base, average='macro'):.3f})  "
          f"rec_mal {recall_score(yte, obf, pos_label=1):.3f}")

obf_eval(lambda t: t, "clean")
obf_eval(add_zero_width, "zero-width")
obf_eval(add_homoglyphs, "гомоглифы")
obf_eval(split_instructions, "разбиение")


**Результат (из прогона):**
- clean: macro-F1 0.735, recall вредоносных 0.953
- **zero-width: macro-F1 0.524, recall вредоносных 0.380** — детектор рушится
- гомоглифы: −0.034 (почти не влияет)
- разбиение: −0.006 (не влияет)

**Вывод.** Детектор ломается не только на новых источниках, но и на простой маскировке —
zero-width-символы разрывают n-граммы и убивают recall вредоносных. Это практический риск:
обфускация дешёвая, а защита — нет.

## 10. Разбор FP/FN: **почему** падает recall безопасных

Это финальный и самый содержательный блок: находим, на ком именно модель ошибается.


In [ ]:
Xtr, ytr, Xte, yte = get_split("source_disjoint")
src = {r["benchmark_id"]: (r["source_name"] or r["source_id"]) for r in prim}
te_ids = [r["benchmark_id"] for r in load_dataset(
    "ProtectSkills/MaliciousSkillBench", "splits", split="source_disjoint")
    if r["split"] in ("test", "validation")]

pipe = make_with(LinearSVC(class_weight="balanced", max_iter=2000))
pipe.fit(Xtr, ytr)
pred = pipe.predict(Xte)

fp = [i for i in range(len(yte)) if yte[i] == 0 and pred[i] == 1]
fn = [i for i in range(len(yte)) if yte[i] == 1 and pred[i] == 0]
tp = [i for i in range(len(yte)) if yte[i] == 1 and pred[i] == 1]

print(f"TP={len(tp)} FP={len(fp)} FN={len(fn)}")
print("recall безопасных:", round(len([i for i in range(len(yte)) if yte[i]==0 and pred[i]==0]) /
      max(1, len([i for i in range(len(yte)) if yte[i]==0])), 3))

print("\nFP по источникам (кого валит зря):")
for s, c in Counter(src[te_ids[i]] for i in fp).most_common(8):
    print(f"  {c:4d}  {s}")


**Результат:** FP=378, FN=62. **359 из 378 FP (95%) — два источника:**
`ATR Skill Security Benchmark` (304/455 = 67% безопасных завалено) и `SkillFortifyBench` (55/90 = 61%).
Остальные источники держатся.

### Почему — top-признаки модели

| FP (ATR) тянут | TP (вредоносные) тянут |
|---|---|
| `the`, `this`, `sdk`, `the user`, `configuration`, `run`, `import`, `python`, `skill`, `name`, `app`, `cache`, `response`, `context`, `setup`, `docs` | `the`, `python`, `python scripts`, `this skill`, `import`, `cache`, `run`, `configuration`, `execute`, `helper`, `the user`, `do not`, `system`, `must` |

**Признаки FP и TP практически совпадают.** Модель не выучила специфичный вредоносный сигнал —
она выучила «технический текст = вредоносный». Специфичных маркеров (`bash`, `eval`, `curl`, `chmod`)
в топе нет.

### Итоговый вывод

1. **Провал не равномерный** — он сконцентрирован в источниках, где безопасные скиллы технически
   насыщены (SDK, CLI, code-review, вызовы инструментов).
2. **Нет специфичного сигнала** — модель опирается на общий технический словарь, поэтому любой новый
   технический скилл лексически неотличим от вредоносного.
3. Это объясняет, почему **char_wb, эмбеддинги и LLM-каскад не помогли**: проблема не в форме и не в
   смысле, а в **переносе лексических маркеров между источниками** и **отсутствии специфичного
   вредоносного признака**.
4. **Что работает** — поведенческие признаки (+0.046 recall безопасных), т.е. сигналы, которые не
   зависят от стиля источника.

### Практический вывод для детекторов скиллов

Детектор, обученный на «техничности текста», будет валить легитимные инструменты — это не edge case,
а системное свойство. Для прода нужны признаки, инвариантные к источнику: что скилл *делает*
(вызовы, права, доступ к сети), а не как он *описан*.
